Block 1: Imports and settings

In [28]:
import json, time, urllib.parse
from datetime import datetime, timedelta
from pathlib import Path

import requests
import feedparser
import pandas as pd

RAW = Path("data/raw_rss")
OUT = Path("data/news_clean_rss.csv")
HEADERS = {"User-Agent": "Mozilla/5.0"}

Block 2: Your stock list

In [29]:
STOCKS = {
    "TCS.NS":        '"Tata Consultancy Services" OR TCS',
    "INFY.NS":       'Infosys',
    "WIPRO.NS":      'Wipro',
    "HCLTECH.NS":    'HCLTech OR "HCL Technologies"',
    "TECHM.NS":      '"Tech Mahindra"',
    "LTM.NS":        'LTIMindtree OR "LTM Limited"',
    "MPHASIS.NS":    'Mphasis',
    "PERSISTENT.NS": '"Persistent Systems"',
    "COFORGE.NS":    'Coforge',
    "LTTS.NS":       '"L&T Technology Services"',
}

Block 3: One RSS request, with retries

In [30]:
def fetch_rss(query, start, end):
    q = f"{query} after:{start:%Y-%m-%d} before:{end:%Y-%m-%d}"
    url = "https://news.google.com/rss/search?" + urllib.parse.urlencode(
        {"q": q, "hl": "en-IN", "gl": "IN", "ceid": "IN:en"})

    for attempt in range(5):
        r = requests.get(url, headers=HEADERS, timeout=30)
        if r.status_code == 200:
            break
        if r.status_code in (429, 503):
            time.sleep(5 * 2 ** attempt)          # back off: 5, 10, 20, 40, 80 s
            continue
        r.raise_for_status()
    else:
        raise RuntimeError("Google is throttling requests. Wait 30+ minutes and re-run.")

    feed = feedparser.parse(r.content)
    arts = []
    for e in feed.entries:
        if not e.get("published"):
            continue
        src = e.get("source", {}).get("title", "")
        title = e.title
        if src and title.endswith(" - " + src):     # Google appends " - Source" to titles
            title = title[: -len(" - " + src)]
        arts.append({
            "publishedAt": pd.to_datetime(e.published, utc=True).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "title": title.strip(),
            "source": src,
            "url": e.link,
        })
    time.sleep(2)                                   # be polite between requests
    return {"articles": arts}

Block 4: Test on one stock and one week (run this first)

In [31]:
test = fetch_rss(STOCKS["INFY.NS"], datetime(2023, 5, 1), datetime(2023, 5, 8))
arts = test["articles"]
print(len(arts), "articles")
print("date range:", min(a["publishedAt"] for a in arts), "to", max(a["publishedAt"] for a in arts))
for a in arts[:10]:
    print(a["publishedAt"], "|", a["source"], "|", a["title"])

17 articles
date range: 2023-05-01T07:00:00Z to 2023-05-08T07:00:00Z
2023-05-02T19:13:19Z | infosys.com | Infosys is a Market Maker in capioIT Salesforce Asia Pacific Systems Integration and Services Providers Capture Share Report - 2023
2023-05-08T07:00:00Z | Business Today | LIC portfolio shares: Reliance Industries, ITC, TCS, Infosys & SBI among insurer's top stock holdings
2023-05-02T07:00:00Z | India CSR | CSR: Infosys Foundation & GoSports Launch 'Girls for Gold' Program for Women Athletes in India
2023-05-08T07:00:00Z | Equitymaster | Infosys vs HDFC Bank: The 10 Year Bet?
2023-05-06T13:26:31Z | infosys.com | Infosys recognized as a leader in Digital Banking Services, U.S. 2022 by ISG
2023-05-01T07:00:00Z | Moneycontrol.com | IT and ChatGPT: Generative AI gains ground among India's top IT firms
2023-05-04T07:00:00Z | Business Today | How Infosys' Nandan Nilekani helped transform India into an IT powerhouse
2023-05-02T07:00:00Z | YourStory.com | Girls for Gold programme strives t

Block 5: Windows with automatic splitting

In [32]:
def windows(start, end, days=30):
    cur = start
    while cur < end:
        nxt = min(cur + timedelta(days=days), end)
        yield cur, nxt
        cur = nxt

CAP = 90     # if a window returns this many or more, it was probably cut off, so split it

def fetch_window(ticker, query, a, b):
    folder = RAW / ticker
    folder.mkdir(parents=True, exist_ok=True)
    f = folder / f"{a:%Y%m%d}_{b:%Y%m%d}.json"

    if f.exists():                                   # cached, no request needed
        data = json.loads(f.read_text(encoding="utf-8"))
    else:
        data = fetch_rss(query, a, b)
        f.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")

    n = len(data["articles"])
    span = (b - a).days
    if n >= CAP:
        if span > 1:
            mid = a + timedelta(days=span // 2)      # split in half and fetch both parts
            fetch_window(ticker, query, a, mid)
            fetch_window(ticker, query, mid, b)
        else:
            print(f"  cap still hit for {ticker} {a:%Y-%m-%d}: some articles missed")

def download(start, end, window_days=30, tickers=None):
    for ticker, query in STOCKS.items():
        if tickers and ticker not in tickers:
            continue
        for a, b in windows(start, end, window_days):
            fetch_window(ticker, query, a, b)
        print(ticker, "done")

Block 6: Small trial run, then the full run

In [34]:
START = datetime(2021, 1, 1)
END = datetime.now()
download(START, END)

  cap still hit for TCS.NS 2025-07-27: some articles missed
  cap still hit for TCS.NS 2025-07-28: some articles missed
  cap still hit for TCS.NS 2025-10-09: some articles missed
  cap still hit for TCS.NS 2026-01-11: some articles missed
  cap still hit for TCS.NS 2026-01-12: some articles missed
  cap still hit for TCS.NS 2026-01-13: some articles missed
  cap still hit for TCS.NS 2026-02-11: some articles missed
  cap still hit for TCS.NS 2026-02-12: some articles missed
  cap still hit for TCS.NS 2026-02-18: some articles missed
  cap still hit for TCS.NS 2026-02-19: some articles missed
  cap still hit for TCS.NS 2026-02-24: some articles missed
  cap still hit for TCS.NS 2026-04-08: some articles missed
  cap still hit for TCS.NS 2026-04-09: some articles missed
  cap still hit for TCS.NS 2026-04-10: some articles missed
  cap still hit for TCS.NS 2026-04-12: some articles missed
  cap still hit for TCS.NS 2026-04-13: some articles missed
  cap still hit for TCS.NS 2026-04-14: s

Check 1: Coverage per stock per year

In [37]:
df = add_trade_date(load_raw())
df["year"] = df["trade_date"].str[:4]
print(df.pivot_table(index="ticker", columns="year", values="title", aggfunc="count", fill_value=0))

year           2021  2022  2023  2024  2025  2026
ticker                                           
COFORGE.NS       47    57   141   279   548  1309
HCLTECH.NS      220   260   364   474   718  2078
INFY.NS         477   619  1038  1276  2395  4317
LTM.NS            2    48   121   184   289   335
LTTS.NS          76    94   112   117   130   290
MPHASIS.NS       47    60    69   108   215   440
PERSISTENT.NS    81   106   110   126   224   798
TCS.NS          602   784  1398  1584  3682  5816
TECHM.NS        231   224   311   364   585  1473
WIPRO.NS        404   458   605   805  1085  2824


Check 2: Are the old windows empty, or never downloaded?

In [38]:
rows = []
for f in sorted((RAW / "INFY.NS").glob("*.json")):
    n = len(json.loads(f.read_text(encoding="utf-8"))["articles"])
    rows.append((f.stem, n))
for r in rows[:30]:
    print(r)

('20210101_20210131', 42)
('20210131_20210302', 31)
('20210302_20210401', 34)
('20210401_20210501', 47)
('20210501_20210531', 28)
('20210531_20210630', 36)
('20210630_20210730', 40)
('20210730_20210829', 47)
('20210829_20210928', 50)
('20210928_20211028', 44)
('20211028_20211127', 34)
('20211127_20211227', 47)
('20211227_20220126', 34)
('20220126_20220225', 35)
('20220225_20220327', 44)
('20220327_20220426', 65)
('20220426_20220526', 41)
('20220526_20220625', 61)
('20220625_20220725', 55)
('20220725_20220824', 42)
('20220824_20220923', 49)
('20220923_20221008', 31)
('20220923_20221023', 94)
('20221008_20221023', 66)
('20221023_20221122', 69)
('20221122_20221222', 49)
('20221222_20230121', 57)
('20230101_20230131', 63)
('20230121_20230220', 45)
('20230131_20230302', 46)


In [41]:
t = fetch_rss(STOCKS["INFY.NS"], datetime(2022, 3, 7), datetime(2022, 3, 14))
print(len(t["articles"]))
for a in t["articles"][:10]:
    print(a["publishedAt"], "|", a["source"], "|", a["title"])

15
2022-03-14T06:06:34Z | Infosys | Infosys Featured in Constellation ShortList™ for Microsoft End-to-End Service Providers
2022-03-07T08:00:00Z | Livemint | Infosys vs Wipro: Which IT stock is better?
2022-03-08T08:00:00Z | YourStory.com | From the army trenches to upskilling herself for the corporate world, meet Capt Shanthi S of Infosys
2022-03-11T08:00:00Z | Business Today | How Salil Parekh Turned Around Infosys
2022-03-09T15:21:47Z | Infosys | Lorene Miller
2022-03-11T08:00:00Z | Business Insider | Equity and diversity in the workplace is a priority for sustainable finance, and the companies it serves
2022-03-11T08:00:00Z | NDTV Profit | Stocks To Watch: Cipla, JSPL, Wipro, Indraprastha Gas, Sun Pharma, Tata Steel
2022-03-11T08:00:00Z | YourStory.com | Shalini Chaudhari of Infosys on diversity and inclusion, nurturing women in tech, and making happiness pie charts to navigate challenges
2022-03-09T11:11:35Z | Infosys | Mary Flipse
2022-03-10T08:00:00Z | Livemint | TWICH+ by Varun

Block 7: Load the cache and remove duplicates

In [36]:
def load_raw():
    rows = []
    for f in RAW.glob("*/*.json"):
        ticker = f.parent.name
        for art in json.loads(f.read_text(encoding="utf-8")).get("articles", []):
            src = art.get("source", "")
            if isinstance(src, dict):                 # GNews format
                src = src.get("name", "")
            rows.append({
                "ticker": ticker,
                "published_utc": art["publishedAt"],
                "title": art.get("title", ""),
                "description": art.get("description", ""),
                "source": src,
                "url": art.get("url", ""),
            })
    df = pd.DataFrame(rows)
    return df.drop_duplicates(["ticker", "url"]).drop_duplicates(["ticker", "title"])

Block 8: Convert to IST and assign the trading date
"Market holidays aren't handled here. Later, map each trade_date to the next date in your yfinance price index."

In [42]:
def add_trade_date(df):
    ist = pd.to_datetime(df["published_utc"], utc=True).dt.tz_convert("Asia/Kolkata")
    after_close = (ist.dt.hour * 60 + ist.dt.minute) >= 15 * 60 + 30     # after 15:30 IST
    d = ist.dt.normalize().dt.tz_localize(None) + pd.to_timedelta(after_close.astype(int), unit="D")
    wd = d.dt.weekday
    d = d + pd.to_timedelta((wd == 5) * 2 + (wd == 6) * 1, unit="D")     # Sat/Sun -> Monday
    df["published_ist"] = ist.dt.strftime("%Y-%m-%d %H:%M")
    df["trade_date"] = d.dt.strftime("%Y-%m-%d")
    df["text"] = df["title"].fillna("").str.strip()
    return df.sort_values(["ticker", "published_ist"])

In [43]:
df = add_trade_date(load_raw())
before = len(df)
df = df[df["title"].str.split().str.len() >= 4]       # drops name-only pages
print(before, "->", len(df))

44034 -> 43896


Block 9: Build and save the dataset

In [44]:
df = add_trade_date(load_raw())
OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT, index=False, encoding="utf-8-sig")
print(len(df), "articles saved to", OUT)

44034 articles saved to data\news_clean_rss.csv


Block 10: Coverage check (this decides whether the data is usable)

In [45]:
df["year"] = df["trade_date"].str[:4]
print(df.pivot_table(index="ticker", columns="year", values="title", aggfunc="count", fill_value=0))
print()
print(df.groupby("source").size().sort_values(ascending=False).head(10))
print()
print(df.sample(10, random_state=0)[["ticker", "published_ist", "title"]])

year           2021  2022  2023  2024  2025  2026
ticker                                           
COFORGE.NS       47    57   141   279   548  1309
HCLTECH.NS      220   260   364   474   718  2078
INFY.NS         477   619  1038  1276  2395  4317
LTM.NS            2    48   121   184   289   335
LTTS.NS          76    94   112   117   130   290
MPHASIS.NS       47    60    69   108   215   440
PERSISTENT.NS    81   106   110   126   224   798
TCS.NS          602   784  1398  1584  3682  5816
TECHM.NS        231   224   311   364   585  1473
WIPRO.NS        404   458   605   805  1085  2824

source
Livemint                     2344
Moneycontrol.com             2202
economictimes.com            2193
Business Standard            2107
Business Today               1774
The Times of India           1505
NDTV Profit                  1107
financialexpress.com         1000
CNBC TV18                     946
Tata Consultancy Services     898
dtype: int64

            ticker     published_ist  

In [47]:
import pandas as pd

df = pd.read_csv("data/news_clean_rss.csv")
print(len(df))          # total rows
# or, if you want to exclude the header automatically (read_csv already does):
print(df.shape[0])      # same thing, via shape

44034
44034
